This notebook generates n simulated replicates of the shendure dataset. As though the whole experiment were run n more times. 

Imports

In [1]:
import scMPRAforge as scm
from dask.distributed import Client, LocalCluster

2025-11-11 10:19:43.663360: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-11 10:19:43.666703: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [ ]:
%load_ext autoreload
%autoreload 2

Set up cluster

In [ ]:
cluster=LocalCluster(memory_limit='8GB')
client=Client(cluster)

2025-11-11 10:27:37,239 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:34077' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'lambda-8b0603407ace033f883e7a992b36cd08', 'lambda-e652c1ce5d5f9f942059c2de2a79d6d6', 'lambda-1cbeeb7dfb50490ff79917263efbe1a5', 'lambda-692c1deebe4af2d4290cd03f0c3fe372', 'lambda-998236f60e1d15b0b7553255cb58be73', 'lambda-fc6712c277e1222db2b8a98e4c78c1b7', 'lambda-f33b8754a91d03451f1449d6e3502ba2', 'lambda-f779afc13fd8f79e267dbcd97c57eacb', 'lambda-0cf08a330482229c42696883dba38505', 'lambda-11c6ffaf17b450a929de9f590fb4c186', 'lambda-9a23d9c814d6775c78c91e333ddafb78', 'lambda-7166d4de39108cbbcdce6e96d49c4db4', 'lambda-b6f2872d772682126c7318f8f7837a2e', 'lambda-cd6b6af60ff93d9cb202996e519da161', 'lambda-e7dd053e69a9fcd8b7a2662422e98b7d', 'lambda-480adc52a88b81f03ebafe22c285ef4f', 'lambda-79d13eac7104669d4dc3fe475281e08b', 'lambda-de1def61880b135e97be2503e3095e4c', 'lambda-7117448494b35bc147cf9540c680

In [3]:
client.dashboard_link

'http://127.0.0.1:8787/status'

Load shendure ortho

In [4]:
name="ortho_primordial_intercept"
data_root="/home/mcn26/project_pi_skr2/shared/tabula_data"
primordial=scm.ortho.load(client,f"{data_root}/shendure",name)

Create a ground-truth dataframe from the description.

In [5]:
description=scm.describe_parameters(primordial.by_cell_type_parameters,
                        dat=primordial.training_data.data,
                        split="cell_type")
gt=description[["cell_type","cre_id","mu"]].groupby(["cell_type","cre_id"]).agg(true_mean=("mu","mean")).reset_index()
gt=scm.zero_pad_ground_truth(gt)
gt

,cell_type,cre_id,true_mean
0,Cardiomyocytes,Bend5_chr4_8174,0.063540
1,Cardiomyocytes,Bend5_chr4_8175,1.620998
2,Cardiomyocytes,Bend5_chr4_8179,0.053297
3,Cardiomyocytes,Btg1_chr10_9578,3.487160
4,Cardiomyocytes,Cdk5r1_chr11_12559,0.280126
...,...,...,...
1856,SurfaceEctoderm,Tubb2b_chr13_2578,0.000000
1863,SurfaceEctoderm,Txndc12_chr4_7969,0.000000
1864,SurfaceEctoderm,Txndc12_chr4_7971,0.000000
1866,SurfaceEctoderm,Txndc12_chr4_7975,0.000000


Abstract what the library looked like from the ortho training data. We will make the simplifying assumption that all MPRA bc are equal abundance in the original & none were lost.

In [6]:
library=primordial.training_data.data[["cre_id","mpra_bc"]].drop_duplicates()
abundance = 1/len(library)
library["abundance"]=abundance
library

,cre_id,mpra_bc,abundance
0,Txndc12_chr4_7978,ACGTAACATTATAAT,0.000035
1,Klf4_chr4_3952,TGTTTAAGTCAACAA,0.000035
2,Foxa2_chr2_13840,CAACAACACATTTTA,0.000035
3,reference,TACCTAATGGGAAAG,0.000035
4,Lamc1_chr1_12152,AAATGGTAAAAGGCA,0.000035
...,...,...,...
751918,Epas1_chr17_10116,GGAACTCAGTCCGTT,0.000035
751919,Bend5_chr4_8174,AATTACAACTACCAA,0.000035
758757,Sparc_chr11_7233,GAGCGGGTTCAAGAT,0.000035
777357,Foxa2_chr2_13830,TACAATGCCCATTAT,0.000035


Create object

In [7]:
simu_obj=scm.de_novo_simulation(
                        simulation_replicates=3,
                        experiment_bounds=scm.SHENDURE_BOUNDS,
                        ground_truth=gt,
                        library=library)

Simulate

In [8]:
simu_obj.gamut(client)

In [ ]:
DATA_ROOT="/home/mcn26/project_pi_skr2/shared/tabula_data/"

scMPRAforge: INFO: 2265/256680 cells (0.882%) have ≥1 multi-transfection event.
scMPRAforge: INFO: 2353/256632 cells (0.917%) have ≥1 multi-transfection event.
scMPRAforge: INFO: 2380/256663 cells (0.927%) have ≥1 multi-transfection event.


In [11]:
simu_obj.simulated_scMPRA

[<Future: finished, type: scMPRAforge.core.scMPRA_data, key: _wrap_helper-95948101903237eaedbca117cea37904>,
 <Future: finished, type: scMPRAforge.core.scMPRA_data, key: _wrap_helper-07f198ee578dd06af8dc0fdd01bba0ff>,
 <Future: finished, type: scMPRAforge.core.scMPRA_data, key: _wrap_helper-c34208fca89f49a514ed9388b83541ab>]

In [12]:
simu_obj.save(path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251111")

In [13]:
cluster.close()
client.close()